In [17]:
# Install once per environment
!pip -q install requests beautifulsoup4 lxml nltk textblob pandas scikit-learn joblib

import re, math, json, datetime as dt
import numpy as np, pandas as pd, requests, joblib
from urllib.parse import urlparse
from bs4 import BeautifulSoup
from nltk.corpus import stopwords
import nltk
from textblob import TextBlob

# NLTK data (first run only)
nltk.download('punkt', quiet=True)
nltk.download('stopwords', quiet=True)

STOPWORDS = set(stopwords.words('english'))
RANDOM_STATE = 42

In [18]:
# 1) Load: a) your saved model(s), b) the UCI CSV for schema + default means
# Paths from your training notebook (adjust if you changed names)
MODEL_LR_PATH = "model_logreg_balanced.joblib"
MODEL_RF_PATH = "model_random_forest_balanced.joblib"
CSV_PATH       = "OnlineNewsPopularity.csv"   # same CSV used to train

# Load models (one or both)
model_lr = joblib.load(MODEL_LR_PATH)
model_rf = joblib.load(MODEL_RF_PATH)

# Load UCI dataset to recover exact feature schema & compute default means
raw = pd.read_csv(CSV_PATH)
raw.columns = [c.strip() for c in raw.columns]

# Rebuild the same preprocessing you used before training:
drop_cols = [c for c in raw.columns if c.lower() in ["url", "timedelta"]]
df_schema = raw.drop(columns=drop_cols, errors="ignore").copy()
# build binary target as in training (we'll drop it from schema later)
df_schema["popular"] = (df_schema["shares"] >= 1400).astype(int)
df_schema.drop(columns=["shares"], inplace=True)

FEATURE_COLUMNS = [c for c in df_schema.columns if c != "popular"]
FEATURE_MEANS = df_schema[FEATURE_COLUMNS].mean(numeric_only=True).to_dict()

print("Total features expected by the model:", len(FEATURE_COLUMNS))
print("First 10 columns:", FEATURE_COLUMNS[:10])

Total features expected by the model: 58
First 10 columns: ['n_tokens_title', 'n_tokens_content', 'n_unique_tokens', 'n_non_stop_words', 'n_non_stop_unique_tokens', 'num_hrefs', 'num_self_hrefs', 'num_imgs', 'num_videos', 'average_token_length']


In [19]:
# 2) Lightweight feature extractor from a real article URL

# This computes the overlapping UCI fields and sensibly defaults the rest to dataset means.

def tokenize(text):
    tokens = re.findall(r"\b\w+\b", text.lower())
    return tokens

def count_nonstop(tokens):
    return sum(1 for t in tokens if t not in STOPWORDS)

def get_week_flags(date_obj):
    # Weekday flags like UCI: weekday_is_monday ... weekday_is_friday; plus is_weekend
    # UCI also has weekday_is_saturday, weekday_is_sunday in some distributions.
    # We'll create all seven flags if present in schema.
    flags = {}
    wd = date_obj.weekday()  # 0=Mon ... 6=Sun
    name_map = {
        0: "weekday_is_monday",
        1: "weekday_is_tuesday",
        2: "weekday_is_wednesday",
        3: "weekday_is_thursday",
        4: "weekday_is_friday",
        5: "weekday_is_saturday",
        6: "weekday_is_sunday",
    }
    for i, col in name_map.items():
        if col in FEATURE_COLUMNS:
            flags[col] = 1 if wd == i else 0
    if "is_weekend" in FEATURE_COLUMNS:
        flags["is_weekend"] = 1 if wd in (5,6) else 0
    return flags

def guess_channel_flags(url, title, text):
    """
    Heuristic mapping into UCI one-hot channels, e.g.:
    data_channel_is_lifestyle, data_channel_is_entertainment, data_channel_is_bus,
    data_channel_is_socmed, data_channel_is_tech, data_channel_is_world
    """
    channel_cols = [c for c in FEATURE_COLUMNS if c.startswith("data_channel_is_")]
    flags = {c: 0 for c in channel_cols}
    hay = " ".join([url.lower(), title.lower(), text.lower()])

    keywords = {
        "data_channel_is_entertainment": ["entertain", "movie", "film", "music", "celebrity", "bollywood", "hollywood", "show"],
        "data_channel_is_tech": ["tech", "software", "gadget", "ai", "ml", "device", "startup", "engineering"],
        "data_channel_is_bus": ["business", "market", "stock", "finance", "economy", "trade", "company"],
        "data_channel_is_world": ["world", "international", "global", "war", "country", "diplomacy"],
        "data_channel_is_socmed": ["twitter", "facebook", "instagram", "whatsapp", "social media", "tiktok", "reddit"],
        "data_channel_is_lifestyle": ["lifestyle", "health", "travel", "food", "fitness", "fashion"],
    }
    # choose the first matching channel (you can refine)
    for col, words in keywords.items():
        if col in flags and any(w in hay for w in words):
            flags[col] = 1
            break
    return flags

def safe_float(val, default=0.0):
    try:
        return float(val)
    except Exception:
        return float(default)

def extract_features_from_url(url: str):
    # Fetch HTML
    resp = requests.get(url, timeout=30)
    resp.raise_for_status()
    html = resp.text
    soup = BeautifulSoup(html, "lxml")

    # Title
    title = (soup.title.string or "").strip() if soup.title else ""
    og_title = soup.find("meta", property="og:title")
    if og_title and og_title.get("content"):
        title = og_title["content"].strip() or title

    # Article text (very rough): all <p> text
    paragraphs = [p.get_text(" ", strip=True) for p in soup.find_all("p")]
    text = " ".join(paragraphs)
    # Fallback: meta description
    if len(text) < 300:
        meta_desc = soup.find("meta", attrs={"name": "description"}) or soup.find("meta", property="og:description")
        if meta_desc and meta_desc.get("content"):
            text += " " + meta_desc["content"]

    # Basic counts
    t_tokens = tokenize(text)
    title_tokens = tokenize(title)
    n_tokens_content = len(t_tokens)
    n_tokens_title = len(title_tokens)
    n_unique_tokens = len(set(t_tokens)) / max(1, n_tokens_content)
    n_non_stop_words = count_nonstop(t_tokens) / max(1, n_tokens_content)
    n_non_stop_unique_tokens = len({t for t in t_tokens if t not in STOPWORDS}) / max(1, n_tokens_content)

    # Links & self-links
    anchors = soup.find_all("a")
    num_hrefs = len(anchors)
    domain = urlparse(url).netloc
    num_self_hrefs = sum(1 for a in anchors if a.get("href") and urlparse(a.get("href")).netloc == domain)

    # Media
    num_imgs = len(soup.find_all("img"))
    # Videos: crude count of <video>, <iframe> pointing to YT/Vimeo, etc.
    num_videos = len(soup.find_all("video")) + sum(
        1 for f in soup.find_all("iframe") if f.get("src") and any(k in f["src"].lower() for k in ["youtube", "vimeo"])
    )

    # Sentiment proxies ~ the UCI dataset has polarity/subjectivity; we approximate via TextBlob
    blob = TextBlob(text[:100000])  # cap length
    global_sentiment_polarity = safe_float(blob.polarity, 0.0)
    global_subjectivity = safe_float(blob.subjectivity, 0.0)

    # Title sentiment proxy
    tblob = TextBlob(title)
    title_sentiment_polarity = safe_float(tblob.polarity, 0.0)
    title_subjectivity = safe_float(tblob.subjectivity, 0.0)

    # Publication date → weekday flags
    # try meta tags
    pub = soup.find("meta", attrs={"property": "article:published_time"}) or soup.find("meta", attrs={"name": "pubdate"})
    if pub and pub.get("content"):
        try:
            pub_dt = dt.datetime.fromisoformat(pub["content"].replace("Z","+00:00")).date()
        except Exception:
            pub_dt = dt.date.today()
    else:
        pub_dt = dt.date.today()

    week_flags = get_week_flags(pub_dt)
    channel_flags = guess_channel_flags(url, title, text)

    # UCI has other keyword-based stats (kw_min_max, kw_avg_avg, etc.). We’ll plug dataset means as neutral defaults.
    features = {col: FEATURE_MEANS.get(col, 0.0) for col in FEATURE_COLUMNS}

    # Fill what we computed
    def set_if(col, val):
        if col in features: features[col] = val

    set_if("n_tokens_title", n_tokens_title)
    set_if("n_tokens_content", n_tokens_content)
    set_if("n_unique_tokens", n_unique_tokens)
    set_if("n_non_stop_words", n_non_stop_words)
    set_if("n_non_stop_unique_tokens", n_non_stop_unique_tokens)
    set_if("num_hrefs", num_hrefs)
    set_if("num_self_hrefs", num_self_hrefs)
    set_if("num_imgs", num_imgs)
    set_if("num_videos", num_videos)
    set_if("global_subjectivity", global_subjectivity)
    set_if("global_sentiment_polarity", global_sentiment_polarity)
    set_if("title_sentiment_polarity", title_sentiment_polarity)
    set_if("title_subjectivity", title_subjectivity)

    # Some engineered combos used in the paper (if they exist in your schema)
    if "img_vid_sum" in features:
        features["img_vid_sum"] = features.get("num_imgs",0) + features.get("num_videos",0)
    if "title_density" in features:
        features["title_density"] = n_tokens_title / max(1, n_tokens_content)

    # apply weekday flags & channel flags present in your model schema
    for k,v in {**week_flags, **channel_flags}.items():
        if k in features:
            features[k] = v

    # Return as 1-row DataFrame in the exact model column order
    row = pd.DataFrame([[features[c] for c in FEATURE_COLUMNS]], columns=FEATURE_COLUMNS)
    meta_preview = {
        "title": title[:120],
        "pub_date": str(pub_dt),
        "url": url,
        "tokens_title": n_tokens_title,
        "tokens_content": n_tokens_content,
        "num_imgs": num_imgs,
        "num_videos": num_videos,
        "num_hrefs": num_hrefs,
    }
    return row, meta_preview

In [20]:
# 3) Predict with your saved model(s)
def predict_popularity(url, which="rf"):
    X_new, meta = extract_features_from_url(url)

    if which.lower() in ["rf", "random_forest"]:
        model = model_rf
        name = "Random Forest"
    else:
        model = model_lr
        name = "Logistic Regression"

    # Both of your models were trained on the FEATURE_COLUMNS order.
    # If your LR was a pipeline with StandardScaler, it will work transparently here.
    proba = model.predict_proba(X_new)[0,1] if hasattr(model, "predict_proba") else None
    pred = model.predict(X_new)[0]

    print(f"\n=== {name} Prediction ===")
    print("Article:", meta["title"])
    print("URL:", meta["url"])
    print("Published:", meta["pub_date"])
    print(f"Predicted label: {'POPULAR' if pred==1 else 'NOT POPULAR'}")
    if proba is not None:
        print(f"Probability of POPULAR: {proba:.3f}")
    print("\nQuick meta:", {k:v for k,v in meta.items() if k not in ['url']})
    return pred, proba

# Example:
# pred, proba = predict_popularity("https://blog.google/technology/ai/", which="rf")

In [21]:
# 4) Batch test a few URLs (handy for your presentation)
test_urls = [
    # Add any public articles here:
    "https://www.wku.edu/news/articles/index.php?view=article&articleid=12722",
    "https://indianexpress.com/article/sports/cricket/suryakumar-yadav-interview-asia-cup-2025-ind-vs-pak-final-dubai-10278785/?ref=breaking_hp",
    "https://www.thehindu.com/incoming/book-review-my-friends-author-fredrik-backman-novel-art-self-discovery/article70069476.ece",
    "https://x.com/ANI/status/1972697609035731196",
]

results = []
for u in test_urls:
    try:
        pred, proba = predict_popularity(u, which="rf")
        results.append({"url": u, "rf_pred": int(pred), "rf_proba": float(proba) if proba is not None else None})
    except Exception as e:
        print(f"Failed on {u}: {e}")

for u in test_urls:
    try:
        pred, proba = predict_popularity(u, which="lr")
        results.append({"url": u, "rr_pred": int(pred), "rr_proba": float(proba) if proba is not None else None})
    except Exception as e:
        print(f"Failed on {u}: {e}")

pd.DataFrame(results)


=== Random Forest Prediction ===
Article: Decoding Addiction: Alejandro Ramirez Explores the Science of Substance Use
URL: https://www.wku.edu/news/articles/index.php?view=article&articleid=12722
Published: 2025-09-29
Predicted label: POPULAR
Probability of POPULAR: 0.550

Quick meta: {'title': 'Decoding Addiction: Alejandro Ramirez Explores the Science of Substance Use', 'pub_date': '2025-09-29', 'tokens_title': 10, 'tokens_content': 926, 'num_imgs': 7, 'num_videos': 0, 'num_hrefs': 363}

=== Random Forest Prediction ===
Article: Suryakumar Yadav interview: ‘It was very important to take a stand (against Pakistan)… Everyone in the team thought the 
URL: https://indianexpress.com/article/sports/cricket/suryakumar-yadav-interview-asia-cup-2025-ind-vs-pak-final-dubai-10278785/?ref=breaking_hp
Published: 2025-09-29
Predicted label: POPULAR
Probability of POPULAR: 0.700

Quick meta: {'title': 'Suryakumar Yadav interview: ‘It was very important to take a stand (against Pakistan)… Everyone 

,url,rf_pred,rf_proba,rr_pred,rr_proba
0,https://www.wku.edu/news/articles/index.php?vi...,1.0,0.550000,NaN,NaN
1,https://indianexpress.com/article/sports/crick...,1.0,0.700000,NaN,NaN
2,https://www.thehindu.com/incoming/book-review-...,1.0,0.646667,NaN,NaN
3,https://x.com/ANI/status/1972697609035731196,1.0,0.550000,NaN,NaN
4,https://www.wku.edu/news/articles/index.php?vi...,NaN,NaN,1.0,0.958607
5,https://indianexpress.com/article/sports/crick...,NaN,NaN,0.0,0.152163
6,https://www.thehindu.com/incoming/book-review-...,NaN,NaN,0.0,0.239227
7,https://x.com/ANI/status/1972697609035731196,NaN,NaN,0.0,0.365027
